# 🔷 Python Generators, Iterators & Context Managers — Complete Reference Guide

How generators and iterators work under the hood, the three communication
channels that power advanced patterns (`yield`, `send`, `return`), and how
context managers guarantee resource cleanup — with every pattern mapped to
real-world usage in the v8.1 GenAI-First roadmap.

---

## Table of Contents

1. [The Iterator Protocol](#1-the-iterator-protocol)
2. [Generator Functions](#2-generator-functions)
3. [The Three Channels — yield, send, return](#3-the-three-channels--yield-send-return)
4. [Iterator vs Generator Type Hints](#4-iterator-vs-generator-type-hints)
5. [Context Managers — with, __enter__, __exit__](#5-context-managers--with-enter-exit)
6. [@contextmanager Decorator](#6-contextmanager-decorator)
7. [@contextmanager Internal Mechanics](#7-contextmanager-internal-mechanics)
8. [Practical Patterns](#8-practical-patterns)
9. [Common Pitfalls and Gotchas](#9-common-pitfalls-and-gotchas)
10. [ParamSpec + TypeVar — Type-Safe Decorators](#10-paramspec--typevar--type-safe-decorators)
11. [typing vs collections.abc](#11-typing-vs-collectionsabc)
12. [Decision Guide and Quick Reference](#12-decision-guide-and-quick-reference)

---

## 1. The Iterator Protocol

An **iterator** is any object that implements two dunder methods:

| Method | Role |
|---|---|
| `__iter__(self)` | Returns `self` — the object is its own iterator |
| `__next__(self)` | Returns the next value, or raises `StopIteration` when done |

An **iterable** is anything that has `__iter__()` — it produces an iterator
when you call `iter()` on it. A list is *iterable* but is not itself an
iterator; `iter(my_list)` returns a `list_iterator` that is.

**How `for x in obj` works under the hood:**

```
for x in something:          Python desugars this into:
    use(x)
                             iter_obj = something.__iter__()
                             while True:
                                 try:
                                     x = iter_obj.__next__()
                                     use(x)
                                 except StopIteration:
                                     break
```

**ASCII — iterator protocol:**

```
┌───────────────────────────────────────────────────────────────┐
│                   ITERATOR PROTOCOL                           │
│───────────────────────────────────────────────────────────────│
│                                                               │
│  ITERABLE  — has __iter__()                                   │
│  ┌────────────────────────────────────────────┐              │
│  │  ITERATOR  — has __iter__() AND __next__() │              │
│  │  ┌──────────────────────────────────────┐  │              │
│  │  │  GENERATOR  — created by yield       │  │              │
│  │  └──────────────────────────────────────┘  │              │
│  └────────────────────────────────────────────┘              │
│                                                               │
│  for x in iterable:                                           │
│      it = iter(iterable)     → calls iterable.__iter__()     │
│      while True:                                              │
│          x = next(it)        → calls it.__next__()           │
│          ...except StopIteration: break                       │
└───────────────────────────────────────────────────────────────┘
```

In [1]:
# =============================================================================
# MANUAL ITERATION — WHAT for DOES INTERNALLY
# =============================================================================
from __future__ import annotations   # PEP 604 unions + forward refs for the whole session

numbers = [10, 20, 30]

# Step 1: get the iterator object from the iterable
iter_obj = iter(numbers)          # calls numbers.__iter__()
print(f"iterator object : {iter_obj}")
print(f"type            : {type(iter_obj)}")

# Step 2: call __next__() repeatedly
print(f"next() call 1   : {next(iter_obj)}")   # 10
print(f"next() call 2   : {next(iter_obj)}")   # 20
print(f"next() call 3   : {next(iter_obj)}")   # 30

# Step 3: StopIteration once exhausted
try:
    next(iter_obj)
except StopIteration:
    print("StopIteration raised — iterator is exhausted")

iterator object : <list_iterator object at 0x104918430>
type            : <class 'list_iterator'>
next() call 1   : 10
next() call 2   : 20
next() call 3   : 30
StopIteration raised — iterator is exhausted


In [2]:
# =============================================================================
# CUSTOM ITERATOR CLASS — COUNTDOWN
# =============================================================================

class Countdown:
    """Iterator that counts down from a starting number to 1.

    Implements the full iterator protocol:
      - __iter__() returns self (object is its own iterator)
      - __next__() returns next value or raises StopIteration

    Parameters
    ----------
    start : int
        The number to count down from.

    Examples
    --------
    >>> list(Countdown(3))
    [3, 2, 1]
    """

    def __init__(self, start: int) -> None:
        self._current = start

    def __iter__(self) -> Countdown:
        return self                        # iterator is its own iterable

    def __next__(self) -> int:
        if self._current <= 0:
            raise StopIteration           # signal exhaustion to the for loop
        value = self._current
        self._current -= 1
        return value


for num in Countdown(5):
    print(f"  countdown: {num}")

print(f"\nlist(Countdown(3)) = {list(Countdown(3))}")

  countdown: 5
  countdown: 4
  countdown: 3
  countdown: 2
  countdown: 1

list(Countdown(3)) = [3, 2, 1]


In [3]:
# =============================================================================
# ITERABLE vs ITERATOR DISTINCTION
# =============================================================================

my_list = [1, 2, 3]

# A list is ITERABLE — it has __iter__ but NOT __next__
print(f"list has __iter__ : {hasattr(my_list, '__iter__')}")   # True
print(f"list has __next__ : {hasattr(my_list, '__next__')}")   # False ← not an iterator!

# iter() creates an iterator FROM the iterable
my_iter = iter(my_list)
print(f"\niter has __iter__ : {hasattr(my_iter, '__iter__')}")  # True
print(f"iter has __next__ : {hasattr(my_iter, '__next__')}")    # True ← now it's an iterator

# Iterators are also iterable (can be used in for loops directly)
# This is why the same object works both as `for x in it:` and `next(it)`.
print(f"\niter is iterable  : {hasattr(my_iter, '__iter__')}")

list has __iter__ : True
list has __next__ : False

iter has __iter__ : True
iter has __next__ : True

iter is iterable  : True


---

## 2. Generator Functions

A **generator function** uses `yield` instead of `return`. When called, it
does NOT execute its body — it returns a **generator object**. Each call to
`next()` runs the code until the next `yield`, then pauses with all local
state frozen in place. This is how generators achieve constant memory
regardless of data size.

**ASCII — generator pause/resume:**

```
┌───────────────────────────────────────────────────────────────┐
│                  GENERATOR FUNCTIONS                          │
│───────────────────────────────────────────────────────────────│
│                                                               │
│  def my_gen():         gen = my_gen()                         │
│      yield 1  ─────→  next(gen) → executes to yield 1 → ①  │
│      yield 2  ─────→  next(gen) → resumes to yield 2  → ②  │
│      yield 3  ─────→  next(gen) → resumes to yield 3  → ③  │
│               ─────→  next(gen) → StopIteration             │
│                                                               │
│  KEY: Code between yields is FROZEN.                          │
│  Local variables + execution position — all preserved.        │
│  This is how generators stream huge datasets in constant mem. │
└───────────────────────────────────────────────────────────────┘
```

In [4]:
# =============================================================================
# GENERATOR vs REGULAR FUNCTION
# =============================================================================
from collections.abc import Generator, Iterator

def get_squares_list(n: int) -> list[int]:
    """Regular function — builds the entire list in memory before returning.

    Parameters
    ----------
    n : int
        Number of squares to generate.

    Returns
    -------
    list[int]
        All n squares at once.
    """
    result: list[int] = []
    for i in range(n):
        result.append(i * i)
    return result                  # returns everything at once


def get_squares_gen(n: int) -> Generator[int, None, None]:
    """Generator function — yields one square at a time, on demand.

    Parameters
    ----------
    n : int
        Number of squares to generate.

    Yields
    ------
    int
        Next square in the sequence.
    """
    for i in range(n):
        yield i * i                # pauses here, resumes on next()


list_result = get_squares_list(5)
gen_result  = get_squares_gen(5)

print(f"list type : {type(list_result)}")   # <class 'list'>
print(f"gen  type : {type(gen_result)}")    # <class 'generator'>
print(f"list      : {list_result}")         # [0, 1, 4, 9, 16] — all values already
print(f"gen       : {gen_result}")          # <generator object ...> — nothing computed yet

print(f"\nFirst next(): {next(gen_result)}")  # 0 — computes only this one
print(f"Second      : {next(gen_result)}")   # 1
print(f"Third       : {next(gen_result)}")   # 4

list type : <class 'list'>
gen  type : <class 'generator'>
list      : [0, 1, 4, 9, 16]
gen       : <generator object get_squares_gen at 0x10ced13c0>

First next(): 0
Second      : 1
Third       : 4


In [5]:
# =============================================================================
# MEMORY EFFICIENCY — list vs generator
# =============================================================================

import sys

# List comprehension — ALL 1M values in RAM at once
big_list = [x * x for x in range(1_000_000)]

# Generator expression — stores state for ONE value at a time
big_gen = (x * x for x in range(1_000_000))

print(f"list memory : {sys.getsizeof(big_list):>10,} bytes  (full 1M values)")
print(f"gen  memory : {sys.getsizeof(big_gen):>10,} bytes  (just the state machine)")

list memory :  8,448,728 bytes  (full 1M values)
gen  memory :        208 bytes  (just the state machine)


In [6]:
# =============================================================================
# GENERATOR EXPRESSION — PARENTHESES NOT BRACKETS
# =============================================================================

# List comprehension — eager, builds immediately
squares_list = [x * x for x in range(5)]    # brackets  → list

# Generator expression — lazy, produces nothing yet
squares_gen  = (x * x for x in range(5))    # parens    → generator

print(f"list comprehension  : {squares_list}")
print(f"generator expression: {squares_gen}")
print(f"consume with list() : {list(squares_gen)}")

list comprehension  : [0, 1, 4, 9, 16]
generator expression: <generator object <genexpr> at 0x10ceb2810>
consume with list() : [0, 1, 4, 9, 16]


In [7]:
# =============================================================================
# EXECUTION FLOW — PROVING THE PAUSE
# =============================================================================

def traced_generator() -> Generator[str, None, None]:
    """Generator that narrates its own execution to show the freeze-resume cycle.

    Yields
    ------
    str
        Stage labels ("first", "second", "third") in sequence.
    """
    print("  [gen] Starting... running to first yield")
    yield "first"

    print("  [gen] Resumed...  running to second yield")
    yield "second"

    print("  [gen] Resumed...  running to third yield")
    yield "third"

    print("  [gen] Resumed...  no more yields — finishing")
    # StopIteration raised automatically when the function body ends


gen = traced_generator()
print("Created generator (nothing executed yet)")
print(f"next() → got '{next(gen)}'")
print("--- back in caller ---")
print(f"next() → got '{next(gen)}'")
print("--- back in caller ---")
print(f"next() → got '{next(gen)}'")
print("--- back in caller ---")
try:
    next(gen)
except StopIteration:
    print("StopIteration — generator finished")

Created generator (nothing executed yet)
  [gen] Starting... running to first yield
next() → got 'first'
--- back in caller ---
  [gen] Resumed...  running to second yield
next() → got 'second'
--- back in caller ---
  [gen] Resumed...  running to third yield
next() → got 'third'
--- back in caller ---
  [gen] Resumed...  no more yields — finishing
StopIteration — generator finished


In [8]:
# =============================================================================
# STREAMING WORD EXTRACTION — THE SPELLER PATTERN
# =============================================================================

def extract_words(text: str) -> Iterator[str]:
    """Yield words one at a time from text — constant memory regardless of size.

    This is the streaming pattern used in ``text_processor.py`` (Speller Stage 1):
    instead of building a list of all words, we yield each word as it is found
    and move on. A huge file never needs to fit in RAM all at once.

    Parameters
    ----------
    text : str
        Input text to extract words from.

    Yields
    ------
    str
        Each word, in order of appearance.
    """
    word_buffer: list[str] = []
    for char in text:
        if char.isalpha() or (char == "'" and word_buffer):
            word_buffer.append(char)
        elif word_buffer:
            yield "".join(word_buffer)   # yield completes one word — then pauses
            word_buffer.clear()

    if word_buffer:                      # flush the last word if text lacks trailing delim
        yield "".join(word_buffer)


sample = "The cat's hat sat on the mat"
print(f"Text: '{sample}'")
print("Words extracted (streaming, one at a time):")
for word in extract_words(sample):
    print(f"  → '{word}'")

Text: 'The cat's hat sat on the mat'
Words extracted (streaming, one at a time):
  → 'The'
  → 'cat's'
  → 'hat'
  → 'sat'
  → 'on'
  → 'the'
  → 'mat'


---

## 3. The Three Channels — yield, send, return

A generator has **three communication channels** with its caller:

| Channel | Direction | Mechanism | Type hint position |
|---|---|---|---|
| **yield** | Generator → Caller | `yield value` | `Generator[YieldType, …]` |
| **send** | Caller → Generator | `gen.send(value)` | `Generator[…, SendType, …]` |
| **return** | Generator → Caller (final) | `raise StopIteration(value)` | `Generator[…, …, ReturnType]` |

Most generators only use channel 1 (`yield`). Channels 2 and 3 exist for
advanced patterns like coroutines and the `@contextmanager` decorator.

**ASCII — three channels:**

```
┌───────────────────────────────────────────────────────────────┐
│         GENERATOR THREE COMMUNICATION CHANNELS                │
│───────────────────────────────────────────────────────────────│
│                                                               │
│  ┌──────────┐   ① yield (OUT)    ┌──────────┐               │
│  │          │ ─────────────────→ │          │               │
│  │ GENERATOR│                    │  CALLER  │               │
│  │          │ ←───────────────── │          │               │
│  └──────────┘   ② send  (IN)     └──────────┘               │
│       │                                                       │
│       │ ③ return (FINAL) → StopIteration.value               │
│                                                               │
│  Generator[YieldType, SendType, ReturnType]                   │
│              ↑           ↑          ↑                         │
│           OUT via     IN via      into                        │
│           yield       send()   StopIteration.value            │
└───────────────────────────────────────────────────────────────┘
```

In [9]:
# =============================================================================
# CHANNEL 1 — yield (OUT): values flow FROM generator TO caller
# =============================================================================

def fibonacci(limit: int) -> Generator[int, None, None]:
    """Yield Fibonacci numbers up to (but not including) a limit.

    Parameters
    ----------
    limit : int
        Stop yielding when the next Fibonacci number equals or exceeds this.

    Yields
    ------
    int
        Fibonacci numbers in sequence.
    """
    a, b = 0, 1
    while a < limit:
        yield a            # send value OUT to caller — generator pauses
        a, b = b, a + b

print("Fibonacci numbers < 100:")
fibs = [n for n in fibonacci(100)]
print(f"  {fibs}")

Fibonacci numbers < 100:
  [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


In [10]:
# =============================================================================
# CHANNEL 2 — send (IN): values flow FROM caller INTO generator
# =============================================================================

def running_average() -> Generator[float, float, None]:
    """Maintain a running average via send().

    Each ``.send(value)`` pushes a number IN; the generator yields the
    updated average OUT. The yield expression does BOTH at once:

        received = yield current_avg
        ← receive IN          ↑ send OUT

    Parameters / Yields / Returns are implicit for a send-driven coroutine.

    Yields
    ------
    float
        The running average after each received value.
    """
    total = 0.0
    count = 0
    average = 0.0
    while True:
        value = yield average    # ① send `average` OUT, ② receive `value` IN
        total += value
        count += 1
        average = total / count


avg = running_average()
next(avg)     # "prime" the generator — runs to the first yield
              # IMPORTANT: you MUST call next() before send()
              # Reason: send() needs an active yield to deliver the value to.
              # Without priming, there is no yield expression waiting to receive.

print(f"send(10.0) → average = {avg.send(10.0)}")   # 10.0
print(f"send(20.0) → average = {avg.send(20.0)}")   # 15.0
print(f"send(30.0) → average = {avg.send(30.0)}")   # 20.0
print(f"send(40.0) → average = {avg.send(40.0)}")   # 25.0
avg.close()   # explicitly stop the infinite while True loop

send(10.0) → average = 10.0
send(20.0) → average = 15.0
send(30.0) → average = 20.0
send(40.0) → average = 25.0


In [11]:
# =============================================================================
# CHANNEL 2 DETAILED — tracing send() data flow step by step
# =============================================================================

def traced_send() -> Generator[str, str, None]:
    """Generator with traces to show exactly when send() data arrives.

    Yields
    ------
    str
        Echoed version of what was sent in.
    """
    print("  [gen] Running to first yield...")
    received = yield "ready"
    print(f"  [gen] Received '{received}' from send()")

    print("  [gen] Running to second yield...")
    received = yield f"got: {received}"
    print(f"  [gen] Received '{received}' from send()")

    print("  [gen] Running to third yield...")
    yield f"got: {received}"
    print("  [gen] Finished")


gen = traced_send()
result = next(gen)                         # prime — runs to first yield
print(f"[caller] next()         → '{result}'")

result = gen.send("hello")                 # sends "hello" IN, yield returns "got: hello"
print(f"[caller] send('hello')  → '{result}'")

result = gen.send("world")                 # sends "world" IN, yield returns "got: world"
print(f"[caller] send('world')  → '{result}'")

  [gen] Running to first yield...
[caller] next()         → 'ready'
  [gen] Received 'hello' from send()
  [gen] Running to second yield...
[caller] send('hello')  → 'got: hello'
  [gen] Received 'world' from send()
  [gen] Running to third yield...
[caller] send('world')  → 'got: world'


In [12]:
# =============================================================================
# CHANNEL 3 — return (FINAL): value goes into StopIteration.value
# =============================================================================

def process_items() -> Generator[str, None, int]:
    """Process items, yield progress, return final count.

    The return type is the THIRD slot: Generator[str, None, int].
    The int return value is captured via StopIteration.value — it is NOT
    visible from a plain for loop, only from explicit next()/try-except.

    Yields
    ------
    str
        Status message for each item processed.

    Returns
    -------
    int
        Total number of items processed — goes into StopIteration.value.
    """
    count = 0
    for item in ["apple", "banana", "cherry"]:
        yield f"processing: {item}"    # yield status OUT
        count += 1
    return count                       # FINAL value → StopIteration.value


gen = process_items()
print(f"  {next(gen)}")   # "processing: apple"
print(f"  {next(gen)}")   # "processing: banana"
print(f"  {next(gen)}")   # "processing: cherry"

try:
    next(gen)
except StopIteration as e:
    print(f"  Generator finished — return value: {e.value}")   # 3

  processing: apple
  processing: banana
  processing: cherry
  Generator finished — return value: 3


In [13]:
# =============================================================================
# ALL THREE CHANNELS — accumulator using yield, send, and return
# =============================================================================

def accumulator() -> Generator[float, float, str]:
    """Demonstrate all three channels in one generator.

    Channel 1 (yield OUT)   : current running total
    Channel 2 (send  IN)    : next number to add
    Channel 3 (return FINAL): summary string in StopIteration.value

    Yields
    ------
    float
        Running total after each addition.

    Returns
    -------
    str
        Summary string captured in ``StopIteration.value``.
    """
    total = 0.0
    count = 0
    while True:
        value = yield total         # OUT: total, IN: next number (or None via next())
        if value is None:           # next() implicitly sends None → break condition
            break
        total += value
        count += 1

    return f"Sum={total}, Count={count}"   # FINAL → StopIteration.value


acc = accumulator()
next(acc)                                   # prime → yields 0.0
print(f"  send(5.0) → running total = {acc.send(5.0)}")   # 5.0
print(f"  send(3.0) → running total = {acc.send(3.0)}")   # 8.0
print(f"  send(2.0) → running total = {acc.send(2.0)}")   # 10.0

try:
    next(acc)                               # sends None → breaks → return fires
except StopIteration as e:
    print(f"  Final summary: {e.value}")    # "Sum=10.0, Count=3"

  send(5.0) → running total = 5.0
  send(3.0) → running total = 8.0
  send(2.0) → running total = 10.0
  Final summary: Sum=10.0, Count=3


---

## 4. Iterator vs Generator Type Hints

Choose the type hint based on which channels the generator uses:

| Question | Answer | Type hint |
|---|---|---|
| Does it only yield outward? | Yes (90 % of cases) | `Iterator[YieldType]` |
| Does it use `send()` or a meaningful `return` value? | Yes | `Generator[YieldType, SendType, ReturnType]` |

`Iterator` is a **supertype** of `Generator` — every generator is an iterator,
but not every iterator is a generator (class-based iterators exist too).
`Iterator[Y]` is equivalent to `Generator[Y, None, None]` for type-checking
purposes, but simpler and preferred for one-way streaming.

**ASCII — type hierarchy:**

```
┌──────────────────────────────────────────┐
│               Iterable                   │
│  ┌───────────────────────────────────┐   │
│  │           Iterator                │   │
│  │  ┌────────────────────────────┐   │   │
│  │  │        Generator           │   │   │
│  │  │  (yield + send + return)   │   │   │
│  │  └────────────────────────────┘   │   │
│  └───────────────────────────────────┘   │
└──────────────────────────────────────────┘

Iterator[Y]  ≡  Generator[Y, None, None]   (for type checking)
```

In [14]:
# =============================================================================
# Iterator[str] — ONE-WAY STREAMING (90 % of cases)
# =============================================================================

def greetings(names: list[str]) -> Iterator[str]:
    """Yield greeting strings — only sends data OUT, never receives IN.

    Use ``Iterator[YieldType]`` because:
    - No send() needed
    - No meaningful return value
    - Simpler and more readable type annotation

    Parameters
    ----------
    names : list[str]
        People to greet.

    Yields
    ------
    str
        Greeting for each name.
    """
    for name in names:
        yield f"Hello, {name}!"


for g in greetings(["Alice", "Bob", "Carol"]):
    print(f"  {g}")

  Hello, Alice!
  Hello, Bob!
  Hello, Carol!


In [15]:
# =============================================================================
# Generator[Y, S, R] — BIDIRECTIONAL (send + return needed)
# =============================================================================

def echo_counter() -> Generator[str, str, int]:
    """Echo what's sent in, count total sends; return the count.

    Use ``Generator[str, str, int]`` because:
    - send() pushes data IN (channel 2 is active)
    - return gives a meaningful final count (channel 3 is active)

    Yields
    ------
    str
        Echo of the last sent string.

    Returns
    -------
    int
        Total number of successful send() calls; in StopIteration.value.
    """
    count = 0
    response = "ready"
    while True:
        received = yield response
        if received is None:   # next() sends None → stop
            break
        response = f"echo: {received}"
        count += 1
    return count


gen = echo_counter()
next(gen)                                       # prime
print(f"  send('ping') → {gen.send('ping')}")   # "echo: ping"
print(f"  send('pong') → {gen.send('pong')}")   # "echo: pong"
try:
    next(gen)
except StopIteration as e:
    print(f"  Total sends  : {e.value}")         # 2

  send('ping') → echo: ping
  send('pong') → echo: pong
  Total sends  : 2


In [16]:
# =============================================================================
# WHERE TO IMPORT FROM — collections.abc, not typing
# =============================================================================

# Preferred since Python 3.9+ (source of truth):
#   from collections.abc import Iterator    # for simple one-way generators
#   from collections.abc import Generator   # for send() / return generators
#   from collections.abc import Iterable    # for functions that accept any iterable

# Legacy (still works, but aliases — avoid for new code):
#   from typing import Iterator, Generator, Iterable

# Quick rule:
#   "Does the type represent a runtime container or callable?"  → collections.abc
#   "Is it a type-system concept (Protocol, TypeVar, ParamSpec)?" → typing

print("Import cheatsheet:")
print("  from collections.abc import Iterator    # simple streaming")
print("  from collections.abc import Generator   # send() + return")
print("  from collections.abc import Iterable    # accept any iterable")
print("  from typing import TypeVar, ParamSpec   # type system concepts")

Import cheatsheet:
  from collections.abc import Iterator    # simple streaming
  from collections.abc import Generator   # send() + return
  from collections.abc import Iterable    # accept any iterable
  from typing import TypeVar, ParamSpec   # type system concepts


---

## 5. Context Managers — with, __enter__, __exit__

A **context manager** is any object that implements two dunder methods:

| Method | When it runs | Role |
|---|---|---|
| `__enter__(self)` | When the `with` block **starts** | Setup; return value → `as X` |
| `__exit__(self, exc_type, exc_val, exc_tb)` | When the block **ends** — always | Cleanup; return `False` to propagate exceptions |

The `with` statement **guarantees** `__exit__` is called — even if an exception
fires inside the block. That guarantee makes it the right tool for any resource
that must be released: files, database connections, locks, timers.

**ASCII — the with statement desugared:**

```
with EXPRESSION as VARIABLE:        Python desugars this to:
    BODY
                                    manager = EXPRESSION
                                    VARIABLE = manager.__enter__()
                                    try:
                                        BODY
                                    finally:
                                        manager.__exit__(exc_info)

KEY GUARANTEE: __exit__() runs even if BODY raises an exception.
```

**Mental model — the Bouncer Pattern:**

```
with open("file.txt") as f:    # ① Bouncer opens the door  (__enter__)
    data = f.read()            # ② You are inside the club  (your code)
                               # ③ Bouncer closes the door  (__exit__)
                               #    Door closes EVEN IF you cause trouble
```

In [17]:
# =============================================================================
# CLASS-BASED CONTEXT MANAGER — DatabaseConnection
# =============================================================================
import time
from typing import Any


class DatabaseConnection:
    """Simulated DB connection showing the context manager protocol.

    This is the verbose class-based approach. Section 6 shows how
    ``@contextmanager`` reduces this to ~5 lines.

    Real-world analogues: psycopg2, chromadb.Client(), boto3.resource(),
    sqlalchemy.Session(), mlflow.start_run().

    Parameters
    ----------
    db_name : str
        Logical name of the database being connected to.

    Examples
    --------
    >>> with DatabaseConnection("speller_db") as db:
    ...     print(db.connected)
    True
    """

    def __init__(self, db_name: str) -> None:
        self.db_name = db_name
        self.connected = False

    def __enter__(self) -> DatabaseConnection:
        """Setup phase — called when the `with` block opens.

        Returns
        -------
        DatabaseConnection
            ``self`` — becomes the value after ``as``.
        """
        print(f"  [DB] Connecting to '{self.db_name}'...")
        self.connected = True
        return self                   # this is what `as db` receives

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc_val: BaseException | None,
        exc_tb: Any,
    ) -> bool:
        """Cleanup phase — called when the `with` block ends, always.

        Parameters
        ----------
        exc_type : type[BaseException] | None
            Exception class if one was raised, else None.
        exc_val : BaseException | None
            Exception instance if raised, else None.
        exc_tb : Any
            Traceback object if exception raised, else None.

        Returns
        -------
        bool
            Return False to let exceptions propagate (almost always correct).
            Return True to silently suppress them (rarely wanted).
        """
        print(f"  [DB] Disconnecting from '{self.db_name}'...")
        self.connected = False
        if exc_type is not None:
            print(f"  [DB] Exception was: {exc_val!r}")
        return False   # do NOT swallow exceptions

In [18]:
# =============================================================================
# USING DatabaseConnection — normal path and exception path
# =============================================================================

# --- Normal path: __exit__ called after block completes ---
print("=== Normal path ===")
with DatabaseConnection("speller_db") as db:
    print(f"  [app] connected: {db.connected}")
    print("  [app] doing work...")
print(f"  [app] after with: connected = {db.connected}\n")

# --- Exception path: __exit__ STILL called before exception propagates ---
print("=== Exception path ===")
try:
    with DatabaseConnection("speller_db") as db:
        print("  [app] about to raise...")
        raise ValueError("something broke")
except ValueError:
    print("  [app] exception caught by caller")
print(f"  [app] connection cleaned up: db.connected = {db.connected}")

=== Normal path ===
  [DB] Connecting to 'speller_db'...
  [app] connected: True
  [app] doing work...
  [DB] Disconnecting from 'speller_db'...
  [app] after with: connected = False

=== Exception path ===
  [DB] Connecting to 'speller_db'...
  [app] about to raise...
  [DB] Disconnecting from 'speller_db'...
  [DB] Exception was: ValueError('something broke')
  [app] exception caught by caller
  [app] connection cleaned up: db.connected = False


---

## 6. @contextmanager Decorator

Writing `__enter__` and `__exit__` is verbose. `contextlib.contextmanager`
converts a **generator function** with exactly one `yield` into a full
context manager — no class needed.

**Structure:**

```python
@contextmanager
def my_context(arg):
    # SETUP (before yield)  →  runs during __enter__()
    resource = acquire(arg)

    yield resource           →  resource becomes `as X`; generator PAUSES here

    # CLEANUP (after yield) →  runs during __exit__()
    release(resource)
```

**ASCII — execution flow:**

```
with my_context() as resource:
    │
    ├─ ① generator starts running
    ├─ ② code BEFORE yield executes (setup)
    ├─ ③ yield sends resource to 'as' variable — GENERATOR PAUSES
    ├─ ④ your 'with' block body runs
    ├─ ⑤ generator RESUMES after yield
    └─ ⑥ code AFTER yield executes (cleanup)

If exception in ④:
    @contextmanager calls .throw() on the generator
    exception is re-raised at the yield point
    catch it with try/except around yield for graceful handling
```

In [19]:
# =============================================================================
# BASIC @contextmanager — before/yield/after structure
# =============================================================================
from contextlib import contextmanager
from collections.abc import Generator


@contextmanager
def managed_resource(name: str) -> Generator[str, None, None]:
    """Simple context manager using @contextmanager.

    Everything before yield  = setup (__enter__)
    The yield value          = 'as' variable
    Everything after yield   = cleanup (__exit__)

    Parameters
    ----------
    name : str
        Name of the resource being managed.

    Yields
    ------
    str
        The resource name (becomes the `as` variable).
    """
    print(f"  [setup]   acquiring '{name}'")
    yield name                              # pauses here; caller's 'as' gets this value
    print(f"  [cleanup] releasing '{name}'")


with managed_resource("database") as resource:
    print(f"  [body]    using: {resource}")

  [setup]   acquiring 'database'
  [body]    using: database
  [cleanup] releasing 'database'


In [20]:
# =============================================================================
# TIMER CONTEXT MANAGER — the Speller benchmarks.py pattern
# =============================================================================
from dataclasses import dataclass, field


@dataclass(frozen=True, slots=True)
class BenchmarkResult:
    """Immutable record of a single benchmark timing.

    Attributes
    ----------
    operation : str
        Label for the timed operation.
    elapsed_seconds : float
        Wall-clock time in seconds (from ``time.perf_counter()``).
    metadata : dict[str, Any]
        Optional extra fields for extended reporting.
    """
    operation: str
    elapsed_seconds: float
    metadata: dict[str, Any] = field(default_factory=dict)


@contextmanager
def timer(operation_name: str) -> Generator[dict[str, Any], None, None]:
    """Time a block of code; store BenchmarkResult in a mutable container.

    We yield a dict (mutable), not the BenchmarkResult (frozen), because
    timing data is only available AFTER the block runs — and a dict
    lets the generator write into the object the caller already holds.
    See Section 4 of notebook 27 for the immutable-vs-mutable deep dive.

    Parameters
    ----------
    operation_name : str
        Label stored in the result.

    Yields
    ------
    dict[str, Any]
        Empty on entry; ``"result"`` key (BenchmarkResult) added after block.
    """
    container: dict[str, Any] = {}
    start = time.perf_counter()

    yield container          # caller's 'with' block runs here

    elapsed = time.perf_counter() - start
    container["result"] = BenchmarkResult(
        operation=operation_name,
        elapsed_seconds=elapsed,
    )


with timer("example_operation") as t:
    total = sum(range(1_000_000))

result = t["result"]
print(f"  operation : {result.operation}")
print(f"  elapsed   : {result.elapsed_seconds:.6f}s")
print(f"  sum       : {total}")

  operation : example_operation
  elapsed   : 0.007215s
  sum       : 499999500000


In [21]:
# =============================================================================
# EXCEPTION HANDLING IN @contextmanager — try/finally guarantees cleanup
# =============================================================================

@contextmanager
def safe_resource(name: str) -> Generator[str, None, None]:
    """Context manager with guaranteed cleanup via try/finally.

    Without try/finally, code after yield is SKIPPED on exceptions.
    With try/finally, cleanup runs on BOTH the happy path and error path.

    Parameters
    ----------
    name : str
        Name of the simulated resource.

    Yields
    ------
    str
        The resource name.
    """
    print(f"  [setup]   acquiring '{name}'")
    try:
        yield name
    except Exception as e:
        print(f"  [error]   exception in 'with' block: {e}")
        raise              # re-raise so the caller still sees the error
    finally:
        # finally runs regardless of exception — this is the cleanup guarantee
        print(f"  [cleanup] releasing '{name}' (always runs)")


try:
    with safe_resource("connection") as r:
        print(f"  [body]    using {r}")
        raise RuntimeError("simulated error")
except RuntimeError:
    print("  [caller]  caught the error")

  [setup]   acquiring 'connection'
  [body]    using connection
  [error]   exception in 'with' block: simulated error
  [cleanup] releasing 'connection' (always runs)
  [caller]  caught the error


---

## 7. @contextmanager Internal Mechanics

When `@contextmanager` wraps your generator function, it creates a helper class
that uses the generator protocol internally:

```
__enter__():
    gen = your_generator(*args)    # create the generator object
    value = next(gen)              # run to yield → setup executes
    return value                   # value becomes the 'as' variable

__exit__(no exception):
    next(gen)                      # resume after yield → cleanup executes
    # expects StopIteration        # generator finishes normally

__exit__(with exception):
    gen.throw(exc_type, exc_val)   # inject exception AT the yield point
    # generator can catch it with try/except around yield
    # or let it propagate to the caller
```

This is why `@contextmanager` needs `Generator[Y, None, None]` — it uses
`next()` for the yield channel and `.throw()` for error injection. The `send()`
channel is available but unused by `@contextmanager` itself.

In [22]:
# =============================================================================
# MANUAL SIMULATION — what @contextmanager does under the hood
# =============================================================================

def raw_timer_generator(name: str) -> Generator[dict[str, Any], None, None]:
    """Raw generator without @contextmanager — shows the protocol directly.

    Yields
    ------
    dict[str, Any]
        Mutable container; ``"result"`` key set after the block.
    """
    container: dict[str, Any] = {}
    start = time.perf_counter()
    yield container
    elapsed = time.perf_counter() - start
    container["result"] = f"{name}: {elapsed:.6f}s"


# This is what @contextmanager does internally:
gen = raw_timer_generator("manual_test")

# __enter__: run generator to first yield
container = next(gen)
print(f"  __enter__ returned: {container}")   # empty dict

# --- 'with' block body would run here ---
total = sum(range(500_000))

# __exit__: resume generator after yield
try:
    next(gen)              # runs cleanup code → generator finishes → StopIteration
except StopIteration:
    pass                   # expected — generator is done

print(f"  After __exit__  : {container}")    # now has "result"

  __enter__ returned: {}
  After __exit__  : {'result': 'manual_test: 0.003652s'}


In [23]:
# =============================================================================
# .throw() — how exceptions enter generators from outside
# =============================================================================

def demo_throw() -> Generator[str, None, None]:
    """Generator that catches an injected exception and recovers.

    Yields
    ------
    str
        Status strings showing the throw/recovery flow.
    """
    try:
        yield "waiting"
    except ValueError as e:
        print(f"  [gen] caught inside generator: {e}")
        yield "recovered"           # generator recovers and yields a new value


gen = demo_throw()
print(f"  next()  → '{next(gen)}'")

# .throw() injects an exception AT the yield point — generator sees it as raised
recovered_value = gen.throw(ValueError("test error"))
print(f"  throw() → '{recovered_value}'")

  next()  → 'waiting'
  [gen] caught inside generator: test error
  throw() → 'recovered'


---

## 8. Practical Patterns

Four production patterns that appear throughout the v8.1 roadmap:

| Pattern | Use case | Roadmap project |
|---|---|---|
| **`timed()` decorator** | Time any function, preserve its type signature | Speller `benchmarks.py` |
| **Generator pipeline** | Chain transforms; no intermediate lists | DataVault token streaming |
| **Generator cleanup** | Early exit with `gen.close()` → `GeneratorExit` | PolicyPulse RAG retrieval |
| **Streaming accumulator** | Process items lazily, accumulate stats | FormSense batch inbox scan |

In [24]:
# =============================================================================
# timed() DECORATOR — wraps timer() context manager (type-safe with P + T)
# =============================================================================
from typing import ParamSpec, TypeVar
from collections.abc import Callable
from functools import wraps

P = ParamSpec("P")    # captures the decorated function's parameter types
T = TypeVar("T")      # captures the decorated function's return type


@contextmanager
def timer_cm(name: str) -> Generator[dict[str, Any], None, None]:
    """Reusable timing primitive.

    Yields
    ------
    dict[str, Any]
        Empty on entry; ``"elapsed"`` (float, seconds) set after block.
    """
    container: dict[str, Any] = {}
    start = time.perf_counter()
    yield container
    container["elapsed"] = time.perf_counter() - start


def timed(operation_name: str) -> Callable[[Callable[P, T]], Callable[P, T]]:
    """Three-layer, type-safe timing decorator that wraps timer_cm().

    Structure:
        timed("load")              → returns decorator   (LAYER 1: receives NAME)
        decorator(my_func)         → returns wrapper     (LAYER 2: receives FUNCTION)
        wrapper(*args, **kwargs)   → calls func + times  (LAYER 3: receives ARGUMENTS)

    ``ParamSpec(P)`` + ``TypeVar(T)`` ensure mypy preserves the decorated
    function's exact parameter types and return type — nothing is erased.

    Parameters
    ----------
    operation_name : str
        Label passed to timer_cm for diagnostic output.

    Returns
    -------
    Callable[[Callable[P, T]], Callable[P, T]]
        A decorator that preserves the original function's full type signature.
    """
    def decorator(func: Callable[P, T]) -> Callable[P, T]:
        @wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
            with timer_cm(operation_name) as t:
                result = func(*args, **kwargs)
            wrapper.benchmark = t["elapsed"]   # type: ignore[attr-defined]
            return result
        wrapper.benchmark = None               # type: ignore[attr-defined]
        return wrapper                         # type: ignore[return-value]
    return decorator                           # type: ignore[return-value]


@timed("sum_computation")
def heavy_work(n: int) -> int:
    """Sum integers 0 .. n-1.

    Parameters
    ----------
    n : int
        Upper bound (exclusive).

    Returns
    -------
    int
        Sum of range(n).
    """
    return sum(range(n))


answer = heavy_work(1_000_000)
print(f"  result    : {answer}")
print(f"  benchmark : {heavy_work.benchmark:.6f}s")   # type: ignore[attr-defined]
# After decoration, mypy still knows: heavy_work(n: int) -> int  ← types preserved!

  result    : 499999500000
  benchmark : 0.007485s


In [25]:
# =============================================================================
# GENERATOR PIPELINE — chaining generators like Unix pipes
# =============================================================================

# Each stage is a generator; data flows through without intermediate lists.
# Memory usage is constant regardless of input size.

def read_lines(text: str) -> Iterator[str]:
    """Stage 1: yield lines from text.

    Yields
    ------
    str
        Each line from the input text.
    """
    for line in text.strip().split("\n"):
        yield line


def filter_nonempty(lines: Iterator[str]) -> Iterator[str]:
    """Stage 2: skip blank lines.

    Yields
    ------
    str
        Only non-empty lines.
    """
    for line in lines:
        if line.strip():
            yield line


def to_uppercase(lines: Iterator[str]) -> Iterator[str]:
    """Stage 3: transform to uppercase.

    Yields
    ------
    str
        Each line uppercased.
    """
    for line in lines:
        yield line.upper()


sample_text = """
    hello world

    python generators

    are powerful
"""

# Chain generators — no intermediate list at any stage
# Equivalent to: text | read_lines | filter_nonempty | to_uppercase
pipeline = to_uppercase(filter_nonempty(read_lines(sample_text)))

print("Generator pipeline output:")
for line in pipeline:
    print(f"  → {line}")

Generator pipeline output:
  → HELLO WORLD
  →     PYTHON GENERATORS
  →     ARE POWERFUL


In [26]:
# =============================================================================
# GENERATOR CLEANUP — .close() triggers GeneratorExit
# =============================================================================

def limited_reader() -> Iterator[int]:
    """Generator that handles early exit via GeneratorExit.

    When .close() is called (or the generator is garbage collected),
    Python throws GeneratorExit into the generator at the yield point.
    Catching it lets you run cleanup code (close files, log, etc.)

    Yields
    ------
    int
        Integers starting from 0, indefinitely.
    """
    try:
        for i in range(1_000_000):
            yield i
    except GeneratorExit:
        # Fired by .close() — use this for resource cleanup
        print("  [gen] GeneratorExit caught — closing early, running cleanup")


gen = limited_reader()
for i, val in enumerate(gen):
    if i >= 3:
        gen.close()    # explicitly stop — triggers GeneratorExit inside the generator
        break
    print(f"  got: {val}")

  got: 0
  got: 1
  got: 2
  [gen] GeneratorExit caught — closing early, running cleanup


---

## 9. Common Pitfalls and Gotchas

In [27]:
# =============================================================================
# PITFALL 1 — GENERATOR EXHAUSTION: iterators are single-pass
# =============================================================================

def numbers() -> Iterator[int]:
    """Simple three-value generator.

    Yields
    ------
    int
        1, 2, 3 in sequence.
    """
    yield 1
    yield 2
    yield 3


gen = numbers()
first_pass  = list(gen)   # consumes all values
second_pass = list(gen)   # empty — already exhausted!

print(f"  first pass  : {first_pass}")    # [1, 2, 3]
print(f"  second pass : {second_pass}")   # [] ← EMPTY!
print("  Fix: create a new generator for each pass, or use list() if you need reuse")

  first pass  : [1, 2, 3]
  second pass : []
  Fix: create a new generator for each pass, or use list() if you need reuse


In [28]:
# =============================================================================
# PITFALL 2 — FORGETTING TO PRIME send() GENERATORS
# =============================================================================

def receiver() -> Generator[str, int, None]:
    """Generator that receives integers via send() and yields status strings.

    Yields
    ------
    str
        Acknowledgment of received value.
    """
    while True:
        value = yield "ready"
        print(f"  [gen] received: {value}")


gen = receiver()

# gen.send(42)  ← TypeError! "can't send non-None value to a just-started generator"
# You MUST prime first with next() before send().
# Reason: send() delivers a value to the CURRENT yield expression.
# Without priming, no yield is active yet — there is no destination for the value.

next(gen)        # prime — runs to the first yield expression
gen.send(42)     # now it works
gen.close()      # stop the infinite loop

print("  Fix: always call next(gen) once before gen.send()")

  [gen] received: 42
  Fix: always call next(gen) once before gen.send()


In [29]:
# =============================================================================
# PITFALL 3 — return IN A GENERATOR ends it (does not yield)
# =============================================================================

def tricky() -> Iterator[int]:
    """Demonstrates that return stops a generator without yielding.

    Yields
    ------
    int
        Only 1 and 2 — the 3 after return is never reached.
    """
    yield 1
    yield 2
    return          # ← stops the generator; raises StopIteration
    yield 3         # ← UNREACHABLE — never executed  # noqa: unreachable


result = list(tricky())
print(f"  result: {result}")     # [1, 2] — no 3!

  result: [1, 2]


In [30]:
# =============================================================================
# PITFALL 4 — yield from: clean delegation to a sub-generator
# =============================================================================

def inner_gen() -> Iterator[int]:
    """Sub-generator that yields 1 and 2.

    Yields
    ------
    int
        1, then 2.
    """
    yield 1
    yield 2


def outer_manual() -> Iterator[int]:
    """Manual delegation — verbose loop over the inner generator."""
    for val in inner_gen():
        yield val
    yield 3


def outer_yield_from() -> Iterator[int]:
    """Clean delegation using yield from.

    ``yield from sub`` is equivalent to:
      - forwarding all yields from sub to the caller
      - forwarding all send() values from the caller into sub
      - capturing sub's return value (StopIteration.value) as the expression result
    """
    yield from inner_gen()     # yields every value inner_gen() produces
    yield 3


print(f"  manual    : {list(outer_manual())}")
print(f"  yield from: {list(outer_yield_from())}")

  manual    : [1, 2, 3]
  yield from: [1, 2, 3]


---

## 10. ParamSpec + TypeVar — Type-Safe Decorators

Without `ParamSpec`/`TypeVar`, decorators **erase** the decorated function's
type information. Mypy sees `(*Any, **Any) -> Any` instead of the real signature.

**ASCII — the problem and the fix:**

```
WITHOUT ParamSpec / TypeVar:                  WITH ParamSpec / TypeVar:
─────────────────────────────────────         ─────────────────────────────────────
def timed(name):                              P = ParamSpec("P")
    def decorator(func):  ← no info          T = TypeVar("T")
        def wrapper(*args, **kwargs):
            return func(...)                  def timed(name) -> Callable[[Callable[P,T]], Callable[P,T]]:
        return wrapper                            def decorator(func: Callable[P,T]) -> Callable[P,T]:
    return decorator                                  def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
                                                          return func(...)
@timed("load")                                        return wrapper
def load(path: str) -> bool: ...                  return decorator
                                              return decorator
mypy: load(*Any, **Any) -> Any  ← ERASED!
load(42) → mypy: fine (it's NOT)             @timed("load")
                                              def load(path: str) -> bool: ...

                                              mypy: load(path: str) -> bool ← PRESERVED!
                                              load(42) → mypy: ERROR int != str
```

In [31]:
# =============================================================================
# TypeVar BASICS — captures ONE type and links it through a function
# =============================================================================
from typing import TypeVar as TV_demo

T_demo = TV_demo("T_demo")


def identity(x: T_demo) -> T_demo:
    """Returns exactly what it receives — the return type mirrors the input type.

    ``T_demo`` binds to whatever concrete type flows in at the call site:
      identity(42)    → T_demo = int  → returns int
      identity("hi")  → T_demo = str  → returns str

    Parameters
    ----------
    x : T_demo
        Any value.

    Returns
    -------
    T_demo
        The same value, same type.
    """
    return x


int_result = identity(42)        # mypy knows: int
str_result = identity("hello")   # mypy knows: str
print(f"  identity(42)      = {int_result!r}  (mypy infers int)")
print(f"  identity('hello') = {str_result!r}  (mypy infers str)")

  identity(42)      = 42  (mypy infers int)
  identity('hello') = 'hello'  (mypy infers str)


In [32]:
# =============================================================================
# ParamSpec + TypeVar COMBINED — type-preserving decorator
# =============================================================================
from typing import ParamSpec as PS, TypeVar as TV

P_demo = PS("P_demo")
T_typed = TV("T_typed")


def logged(label: str) -> Callable[[Callable[P_demo, T_typed]], Callable[P_demo, T_typed]]:
    """Type-safe logging decorator — the decorated function's signature is preserved.

    The return type annotation reads as:
    "A function that takes a Callable[P, T] (original) and returns Callable[P, T] (same types)."

    Breaking down ``Callable[[Callable[P, T]], Callable[P, T]]``:
      Callable[                        ← outer: the decorator itself
          [Callable[P_demo, T_typed]], ← input: original function
          Callable[P_demo, T_typed]    ← output: wrapped function — SAME signature
      ]

    Parameters
    ----------
    label : str
        Tag shown in log output.

    Returns
    -------
    Callable[[Callable[P_demo, T_typed]], Callable[P_demo, T_typed]]
        A decorator that wraps any function while preserving its types.
    """
    def decorator(func: Callable[P_demo, T_typed]) -> Callable[P_demo, T_typed]:
        @wraps(func)
        def wrapper(*args: P_demo.args, **kwargs: P_demo.kwargs) -> T_typed:
            print(f"    [{label}] calling {func.__name__}")
            result = func(*args, **kwargs)
            print(f"    [{label}] returned {result!r}")
            return result   # type: T_typed — preserved
        return wrapper      # type: ignore[return-value]
    return decorator        # type: ignore[return-value]


@logged("DEMO")
def add_ints(a: int, b: int) -> int:
    """Add two integers.

    Parameters
    ----------
    a, b : int
        Addends.

    Returns
    -------
    int
        Their sum.
    """
    return a + b


@logged("DEMO")
def greet_person(name: str) -> str:
    """Greet someone.

    Parameters
    ----------
    name : str
        Person's name.

    Returns
    -------
    str
        Greeting string.
    """
    return f"Hello, {name}!"


sum_result   = add_ints(3, 4)         # mypy knows: int
greet_result = greet_person("Manuel") # mypy knows: str
print(f"  add_ints(3, 4)         = {sum_result}")
print(f"  greet_person('Manuel') = {greet_result}")
print(f"  add_ints.__name__      = {add_ints.__name__!r}  (preserved by @wraps)")

    [DEMO] calling add_ints
    [DEMO] returned 7
    [DEMO] calling greet_person
    [DEMO] returned 'Hello, Manuel!'
  add_ints(3, 4)         = 7
  greet_person('Manuel') = Hello, Manuel!
  add_ints.__name__      = 'add_ints'  (preserved by @wraps)


In [33]:
# =============================================================================
# P.args AND P.kwargs — the correct way to forward parameters
# =============================================================================

# Inside the wrapper, you CANNOT write *args: P  (P is not a tuple type).
# ParamSpec exposes TWO special attributes for this purpose:
#
#   def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
#                      ↑                  ↑
#              positional arg types   keyword arg types
#              from the captured P    from the captured P
#
# If the original function is:
#   def load_dict(path: str, verbose: bool = False) -> bool:
#
# Then P.args  captures  : (str,)
# And  P.kwargs captures : {verbose: bool}
# And  T       captures  : bool
#
# Mypy threads these through the wrapper so every call site is checked
# against the original parameter types.

print("P.args and P.kwargs explained:")
print("  def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:")
print("  ← positional args typed from P.args")
print("  ← keyword args  typed from P.kwargs")
print("  ← return        typed from T")

P.args and P.kwargs explained:
  def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
  ← positional args typed from P.args
  ← keyword args  typed from P.kwargs
  ← return        typed from T


In [34]:
# =============================================================================
# THREE-LAYER STRUCTURE — why @deco('param') needs three nested functions
# =============================================================================

# @deco                     → two-layer (deco receives the function directly)
# @deco("param")            → three-layer (Python evaluates @deco("param") first)
#
# Step-by-step for @timed("load"):
#   Step 1: Python calls timed("load")      → gets `decorator`
#   Step 2: Python calls decorator(load_fn) → gets `wrapper`
#   load_fn is now replaced by wrapper in the module namespace
#
# Three-layer template (with ParamSpec/TypeVar):
#
#   def deco(name: str) -> Callable[[Callable[P, T]], Callable[P, T]]:
#       def decorator(func: Callable[P, T]) -> Callable[P, T]:
#           @wraps(func)
#           def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
#               before(name)
#               result = func(*args, **kwargs)
#               after(name)
#               return result
#           return wrapper
#       return decorator

print("Layer structure:")
print("  timed('load')         → Layer 1: receives NAME, returns decorator")
print("  decorator(load_dict)  → Layer 2: receives FUNCTION, returns wrapper")
print("  wrapper(path)         → Layer 3: receives ARGUMENTS, calls func")

Layer structure:
  timed('load')         → Layer 1: receives NAME, returns decorator
  decorator(load_dict)  → Layer 2: receives FUNCTION, returns wrapper
  wrapper(path)         → Layer 3: receives ARGUMENTS, calls func


In [35]:
# =============================================================================
# WHAT mypy CATCHES WITH ParamSpec / TypeVar
# =============================================================================

# After:
#   @timed("load")
#   def load_dict(path: str) -> bool: ...
#
# WITH ParamSpec / TypeVar — mypy catches ALL of these:
#   result: str = load_dict("large")  # ERROR: cannot assign bool to str
#   load_dict(42)                     # ERROR: int is not str (wrong param type)
#   load_dict()                       # ERROR: missing required argument 'path'
#   load_dict("a", "b")               # ERROR: too many positional arguments
#
# WITHOUT ParamSpec / TypeVar — mypy catches NONE of the above:
#   load_dict would appear as (*Any, **Any) -> Any — all type info erased.

print("With ParamSpec/TypeVar, mypy catches all signature violations:")
print("  load_dict(42)          → ERROR: int is not str")
print("  result: str = load_fn  → ERROR: bool != str")
print("  load_dict()            → ERROR: missing 'path'")
print("  load_dict('a', 'b')    → ERROR: too many arguments")

With ParamSpec/TypeVar, mypy catches all signature violations:
  load_dict(42)          → ERROR: int is not str
  result: str = load_fn  → ERROR: bool != str
  load_dict()            → ERROR: missing 'path'
  load_dict('a', 'b')    → ERROR: too many arguments


---

## 11. typing vs collections.abc

**History:** `typing` was added in Python 3.5 as the home for all generic
types. Python 3.9 made `collections.abc` directly subscriptable, so
`Iterator[str]` works without `typing`. Python 3.12 officially marks the
`typing` versions as legacy aliases.

**Rule:** import based on what the thing IS.

```
┌──────────────────────────────────────────────────────────────────┐
│  from collections.abc          │  from typing                   │
│  (container / callable types)  │  (type-system concepts)        │
│────────────────────────────────│────────────────────────────────│
│  Generator                     │  Protocol                      │
│  Iterator                      │  runtime_checkable             │
│  Iterable                      │  TypeVar                       │
│  Callable                      │  ParamSpec                     │
│  Sequence                      │  Any                           │
│  Mapping                       │  Final                         │
│  MutableMapping                │  TypeAlias                     │
│  Set / MutableSet              │  TypedDict                     │
│  Sized / Container             │  Literal                       │
│                                │  Annotated                     │
│  ✓ Source of truth             │  ✓ No equivalent in abc        │
│  ✓ mypy / ruff prefer this     │  ✓ Type system only            │
└──────────────────────────────────────────────────────────────────┘
```

**Why:** `collections.abc` IS the runtime ABC hierarchy — Python's own `isinstance`
checks use it. `typing` re-exports them for backward compatibility only.
Modern linters (ruff E501, mypy strict) prefer `collections.abc`.

In [36]:
# =============================================================================
# BOTH WORK — but one is preferred
# =============================================================================

# OLD WAY (typing re-exports) — still valid, but marks code as pre-3.9 style
from typing import Iterator as TypingIterator, Generator as TypingGenerator

# NEW WAY (collections.abc source) — preferred since Python 3.9+
from collections.abc import Iterator as AbcIterator, Generator as AbcGenerator

# They are the same at runtime — isinstance checks are identical
sample_gen = (x for x in range(3))
print(f"isinstance check (typing)  : {isinstance(sample_gen, TypingGenerator)}")
print(f"isinstance check (abc)     : {isinstance(sample_gen, AbcGenerator)}")
print("Both resolve to the same runtime ABC — prefer collections.abc in new code.")

isinstance check (typing)  : True
isinstance check (abc)     : True
Both resolve to the same runtime ABC — prefer collections.abc in new code.


In [37]:
# =============================================================================
# BUILT-IN TYPES — directly subscriptable since Python 3.9 (no import needed)
# =============================================================================

# Before 3.9, you had to import from typing:
#   from typing import List, Dict, Set, Tuple, Type
# Since 3.9, use the built-in lowercase names directly:

numbers_typed: list[int]        = [1, 2, 3]        # was typing.List[int]
scores_typed:  dict[str, int]   = {"a": 1}          # was typing.Dict[str, int]
ids_typed:     set[str]         = {"x", "y"}        # was typing.Set[str]
point_typed:   tuple[int, int]  = (0, 0)            # was typing.Tuple[int, int]

print("Built-in generic types (Python 3.9+):")
print(f"  list[int]        : {numbers_typed}")
print(f"  dict[str, int]   : {scores_typed}")
print(f"  set[str]         : {ids_typed}")
print(f"  tuple[int, int]  : {point_typed}")
print("NEVER import List, Dict, Set, Tuple from typing — use lowercase built-ins.")

Built-in generic types (Python 3.9+):
  list[int]        : [1, 2, 3]
  dict[str, int]   : {'a': 1}
  set[str]         : {'x', 'y'}
  tuple[int, int]  : (0, 0)
NEVER import List, Dict, Set, Tuple from typing — use lowercase built-ins.


In [38]:
# =============================================================================
# QUICK DECISION — where does each name live?
# =============================================================================

# 1. Is it list, dict, set, tuple, type?
#    → use the built-in directly (no import needed)
#      list[str]  dict[str, int]  set[str]  tuple[int, ...]  type[MyClass]

# 2. Is it Iterator, Generator, Iterable, Callable, Sequence, Mapping?
#    → from collections.abc
#      from collections.abc import Iterator, Generator, Callable

# 3. Is it Protocol, TypeVar, ParamSpec, Any, Final, TypedDict, Literal?
#    → from typing
#      from typing import Protocol, TypeVar, ParamSpec, Any

# 4. Not sure?  Ask: "Does it represent a runtime container or callable?"
#    YES → collections.abc       NO (type concept only) → typing

# Canonical imports for your Speller modules:
print("Canonical imports for Speller:")
print("  benchmarks.py  : from collections.abc import Callable, Generator")
print("                   from typing import Any, ParamSpec, TypeVar")
print("  text_processor : from collections.abc import Iterator")
print("  protocols.py   : from typing import Protocol, runtime_checkable")

Canonical imports for Speller:
  benchmarks.py  : from collections.abc import Callable, Generator
                   from typing import Any, ParamSpec, TypeVar
  text_processor : from collections.abc import Iterator
  protocols.py   : from typing import Protocol, runtime_checkable


---

## 12. Decision Guide and Quick Reference

**When to use each tool:**

| Need to... | Use |
|---|---|
| Stream data one-way (out only) | Generator → `Iterator[Y]` |
| Two-way communication | Generator → `Generator[Y, S, R]` + `send()` |
| Ensure cleanup (files, DB, locks) | Context manager (`with`) |
| Time a code block | `@contextmanager` + `yield` |
| Time a function | Decorator wrapping the CM |
| Chain data transforms, no RAM overhead | Generator pipeline |
| Process huge files without loading all | Generator (constant memory) |
| Stream LLM tokens | `async` Generator |

**Type hint cheat sheet:**

| Pattern | Type hint |
|---|---|
| Simple one-way streaming | `Iterator[YieldType]` |
| send() + meaningful return | `Generator[Yield, Send, Return]` |
| `@contextmanager` body | `Generator[YieldType, None, None]` |
| Generator expression | `Generator[YieldType, None, None]` |
| Class-based iterator | `Iterator[YieldType]` |
| Accepts any iterable | `Iterable[ItemType]` |

**Roadmap module map:**

| Module | Pattern |
|---|---|
| `text_processor.py` | Generator → `Iterator[str]` |
| `benchmarks.py` | `@contextmanager` → `timer()` |
| `benchmarks.py` | Decorator → `timed()` wraps `timer()` |
| `benchmarks.py` | `ParamSpec` + `TypeVar` for type-safe `timed()` |
| `dictionary.py` | Context manager → `with open()` in `load()` |
| `__main__.py` | Context manager → configure logging |

---

## Quick Reference

```python
# ─── iterator protocol ────────────────────────────────────────────────────
class MyIterator:
    def __iter__(self) -> MyIterator: return self
    def __next__(self) -> T:
        if done: raise StopIteration
        return next_value

# ─── generator function (one-way) ────────────────────────────────────────
from collections.abc import Iterator

def my_gen(n: int) -> Iterator[int]:
    for i in range(n):
        yield i                # pauses here, state frozen until next()

# ─── generator with send() ───────────────────────────────────────────────
from collections.abc import Generator

def coroutine() -> Generator[float, float, str]:
    total = 0.0
    while True:
        value = yield total    # OUT: total; IN: next number
        if value is None: break
        total += value
    return f"Sum={total}"      # → StopIteration.value

gen = coroutine()
next(gen)                      # MUST prime before send()
gen.send(5.0)                  # push IN, get OUT

# ─── generator expression ────────────────────────────────────────────────
squares_gen = (x * x for x in range(100))   # lazy, constant memory

# ─── yield from (delegation) ─────────────────────────────────────────────
def outer() -> Iterator[int]:
    yield from inner()         # forwards all yields + send() + return value
    yield 99

# ─── context manager — class-based ───────────────────────────────────────
class MyCM:
    def __enter__(self) -> MyCM:
        setup(); return self
    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        cleanup(); return False   # False = propagate exceptions

with MyCM() as cm:
    use(cm)

# ─── @contextmanager (basic) ─────────────────────────────────────────────
from contextlib import contextmanager

@contextmanager
def my_cm(arg: str) -> Generator[dict, None, None]:
    container: dict = {}
    setup(arg)             # __enter__
    yield container        # value → `as X`; generator PAUSES
    cleanup()              # __exit__  (only reached on no exception!)

# ─── @contextmanager (exception-safe) ────────────────────────────────────
@contextmanager
def safe_cm(arg: str) -> Generator[Resource, None, None]:
    resource = acquire(arg)
    try:
        yield resource
    finally:
        release(resource)  # ALWAYS runs — happy path or exception

# ─── @contextmanager (yield None) ────────────────────────────────────────
@contextmanager
def side_effect_cm() -> Generator[None, None, None]:
    setup()
    yield                  # bare yield — caller omits `as X`
    teardown()

# ─── type-safe timing decorator ──────────────────────────────────────────
from typing import ParamSpec, TypeVar

P = ParamSpec("P")
T = TypeVar("T")

def timed(name: str) -> Callable[[Callable[P, T]], Callable[P, T]]:
    def decorator(func: Callable[P, T]) -> Callable[P, T]:
        @wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
            with timer_cm(name) as t:
                result = func(*args, **kwargs)
            wrapper.benchmark = t["elapsed"]  # type: ignore[attr-defined]
            return result
        wrapper.benchmark = None              # type: ignore[attr-defined]
        return wrapper
    return decorator

# ─── generator pipeline ───────────────────────────────────────────────────
pipeline = stage_3(stage_2(stage_1(source)))   # constant memory, lazy

# ─── generator cleanup ───────────────────────────────────────────────────
gen = my_gen()
gen.close()               # throws GeneratorExit at the yield point

# ─── import cheatsheet ───────────────────────────────────────────────────
from collections.abc import Iterator, Generator, Iterable, Callable
from typing import TypeVar, ParamSpec, Any, Protocol, Final
# list[str], dict[str, int], set[str], tuple[int, ...] → NO import needed (3.9+)

# ─── key rules ────────────────────────────────────────────────────────────
# • Generators: use Iterator[Y] for one-way, Generator[Y,S,R] with send()/return
# • Prime with next() before the first send()
# • Generators are single-pass — create a new one if you need to iterate twice
# • @contextmanager: exactly ONE yield — second yield → RuntimeError
# • Yield a MUTABLE container (dict/list) to share data computed after yield
# • Always try/finally around yield for guaranteed cleanup on exceptions
# • Always @wraps(func) in decorators — preserves __name__, __doc__
# • Return False from __exit__ to propagate exceptions (safe default)
# • time.perf_counter() is monotonic — prefer it over time.time()
# • Dynamic wrapper attributes: value  # type: ignore[attr-defined]
```